# Tuned Churn Model: Error Analysis and Verdict

**Goal:** Tune a model against the baseline, examine false positives and false negatives, group failures, and make a deployment recommendation.

**Important limitation:** `customer_churn.csv` is synthetic. Results are useful for demonstrating an ML workflow, not for claiming real-world business performance.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, precision_score, recall_score, confusion_matrix, classification_report
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder

RANDOM_STATE=42
df=pd.read_csv("customer_churn.csv")
X,y=df.drop(columns="churn"),df["churn"]
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.20,stratify=y,random_state=RANDOM_STATE)
numeric_features=["age","income","tenure_years","monthly_spend","support_calls"]
categorical_features=["gender","plan","contract"]
preprocessor=ColumnTransformer([
 ("numeric",Pipeline([("imputer",SimpleImputer(strategy="median")),("scaler",StandardScaler())]),numeric_features),
 ("categorical",Pipeline([("imputer",SimpleImputer(strategy="most_frequent")),("encoder",OneHotEncoder(handle_unknown="ignore"))]),categorical_features)
])
cv=StratifiedKFold(n_splits=3,shuffle=True,random_state=RANDOM_STATE)


## 1. Baseline
The baseline is Logistic Regression with the same preprocessing. The test set is held out from model selection.

In [ ]:
baseline=Pipeline([("preprocessing",preprocessor),("classifier",LogisticRegression(max_iter=1000,random_state=RANDOM_STATE))])
baseline_cv=cross_validate(baseline,X_train,y_train,cv=cv,scoring={"accuracy":"accuracy","roc_auc":"roc_auc","f1":"f1"})
baseline.fit(X_train,y_train)
baseline_pred=baseline.predict(X_test)
baseline_prob=baseline.predict_proba(X_test)[:,1]
pd.Series({"Mean CV ROC-AUC":baseline_cv["test_roc_auc"].mean(),"Mean CV accuracy":baseline_cv["test_accuracy"].mean(),
 "Test accuracy":accuracy_score(y_test,baseline_pred),"Test ROC-AUC":roc_auc_score(y_test,baseline_prob),
 "Test F1":f1_score(y_test,baseline_pred)}).round(4)


## 2. Tune using training-only cross-validation
The search optimizes ROC-AUC over Logistic Regression regularization/class-weight choices and a small Random Forest grid. Preprocessing remains inside the pipeline for every fold.

In [ ]:
search_pipeline=Pipeline([("preprocessing",preprocessor),("classifier",LogisticRegression(max_iter=1000,random_state=RANDOM_STATE))])
param_grid=[
 {"classifier":[LogisticRegression(max_iter=1000,random_state=RANDOM_STATE)],"classifier__C":[0.03,0.1,0.3,1,3,10],"classifier__class_weight":[None,"balanced"]},
 {"classifier":[RandomForestClassifier(n_estimators=100,random_state=RANDOM_STATE)],"classifier__max_depth":[3,5,None],"classifier__min_samples_leaf":[2,5],"classifier__class_weight":[None,"balanced"]}
]
search=GridSearchCV(search_pipeline,param_grid,scoring="roc_auc",cv=cv,n_jobs=1,refit=True,return_train_score=False)
search.fit(X_train,y_train)
tuned_model=search.best_estimator_
print("Best CV ROC-AUC:",round(search.best_score_,4))
print("Best parameters:",search.best_params_)


## 3. Held-out comparison
Evaluate the test set once after tuning. Do not repeatedly change the model based on test performance.

In [ ]:
tuned_pred=tuned_model.predict(X_test)
tuned_prob=tuned_model.predict_proba(X_test)[:,1]
comparison=pd.DataFrame([
 {"model":"Baseline Logistic Regression","CV ROC-AUC":baseline_cv["test_roc_auc"].mean(),"test accuracy":accuracy_score(y_test,baseline_pred),"test ROC-AUC":roc_auc_score(y_test,baseline_prob),"test F1":f1_score(y_test,baseline_pred),"precision":precision_score(y_test,baseline_pred,zero_division=0),"recall":recall_score(y_test,baseline_pred,zero_division=0)},
 {"model":"Tuned model","CV ROC-AUC":search.best_score_,"test accuracy":accuracy_score(y_test,tuned_pred),"test ROC-AUC":roc_auc_score(y_test,tuned_prob),"test F1":f1_score(y_test,tuned_pred),"precision":precision_score(y_test,tuned_pred,zero_division=0),"recall":recall_score(y_test,tuned_pred,zero_division=0)}
])
display(comparison.round(4))
print("Tuned confusion matrix (rows=actual [0,1], columns=predicted [0,1]):")
print(confusion_matrix(y_test,tuned_pred))
print(classification_report(y_test,tuned_pred,digits=3,zero_division=0))


## 4. Individual errors
False positives mean unnecessary outreach to customers who would not churn. False negatives mean missed churners. Their costs are different.

In [ ]:
error_table=X_test.reset_index(drop=True).copy()
error_table["actual_churn"]=y_test.reset_index(drop=True)
error_table["predicted_churn"]=tuned_pred
error_table["predicted_probability"]=tuned_prob
error_table["error_type"]=np.select([
 (error_table.actual_churn==0)&(error_table.predicted_churn==1),
 (error_table.actual_churn==1)&(error_table.predicted_churn==0)],
 ["False positive","False negative"],default="Correct")
display(error_table[error_table.error_type!="Correct"].sort_values("predicted_probability",ascending=False))


## 5. Grouped error analysis
Groups are descriptive. Small subgroup samples are noisy, and observed differences do not establish causes.

In [ ]:
rows=[]
for feature in ["contract","plan","gender","support_calls"]:
 for value,group in error_table.groupby(feature,dropna=False):
  neg=(group.actual_churn==0).sum(); pos=(group.actual_churn==1).sum()
  fp=((group.actual_churn==0)&(group.predicted_churn==1)).sum()
  fn=((group.actual_churn==1)&(group.predicted_churn==0)).sum()
  rows.append({"feature":feature,"group":str(value),"n":len(group),"actual_positive_rate":group.actual_churn.mean(),
   "accuracy":(group.actual_churn==group.predicted_churn).mean(),"false_positives":fp,"false_negatives":fn,
   "FPR":fp/neg if neg else np.nan,"FNR":fn/pos if pos else np.nan})
display(pd.DataFrame(rows).round(3))


## 6. Failure modes and verdict

**Observed trade-off:** tuning selected the model by CV ROC-AUC. In this run, test accuracy and ROC-AUC increase slightly, but F1 decreases. This is not an unqualified improvement.

- **False positives:** may cause unnecessary retention offers or outreach.
- **False negatives:** miss customers who churn and the opportunity to intervene.
- **Default threshold:** `predict()` uses 0.5. Threshold changes affect FP/FN balance. Choose it using business costs on validation data, not repeated test-set inspection.
- **Group uncertainty:** error rates for small groups are unstable. Investigate patterns with more data before drawing conclusions.
- **Synthetic data:** artificial target-generation rules cannot establish real customer performance.

### Verdict: do not deploy to real customers yet
Use this as an ML engineering demonstration only. Before production, validate on representative real data with a temporal holdout, evaluate calibration and drift, choose a cost-based threshold, assess subgroup errors with adequate sample sizes, and run a monitored pilot.

### Leakage controls
The split occurs before tuning. Imputation, scaling, and encoding are inside the pipeline. GridSearchCV fits preprocessing only on each training fold. The held-out test set is not used for parameter selection.
